# single_image_demo — one photograph at a time

Every other notebook in this project argues in aggregate: a mean ROC-AUC
over 50 folds, a consolidated table of arms. That convinces a reviewer.
Watching what happens to a *single photograph's* predicted probability as
the pixels the model is fed degrade to a blur, then a solid colour patch,
then eleven numbers, then nothing at all — that convinces a room, whichever
way the number actually moves. It does not always move the way you'd
expect (Part 5 says exactly where and why it doesn't), and that is worth
seeing too.

**What this notebook does not do.** It does not re-plot any of the six
experiments' result tables — if a cell here could be written by reading a
CSV of numbers already produced elsewhere, it does not belong here. Every
number and image below is either read directly from another experiment's
cache, or a small logistic-regression probe refit in seconds on that cached
array so a *per-image* out-of-fold prediction is available, not just a
fold-level metric. No image is re-extracted, no model is re-trained, and no
file under `data/raw/images/` is touched.

**Honesty constraint.** Every probability shown comes from a model that
never saw that image during training. This is asserted explicitly, per
image and per condition, before any image is displayed (Part 3).

**Housekeeping.** This notebook's cell outputs are cleared before every
commit — what is committed is code, not a frozen run. Re-running the
notebook top to bottom reproduces every figure and number from the caches
already on disk.

## 1. Imports, config, device check

In [ ]:
import inspect
import logging
import random
import sys
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)
# Cached CNN checkpoints trigger torch's routine "weights_only=False" notice on
# every load; this demo only ever loads checkpoints this project itself wrote.
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")

from experiments.demo.config import Config
from experiments.demo import pipeline as demo

cfg = Config()

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"data_root: {cfg.data_root}")
print(f"fold-split base seed (cfg.seed): {cfg.seed}, repeat_index: {cfg.repeat_index}")
print(f"image-selection seed (cfg.selection_seed): {cfg.selection_seed}")

## 2. Image selection, seed, and typicality

### 2.1 Which images can this demo honestly show?

The demo sequence's last row is the fine-tuned CNN, whose out-of-fold
predictions only exist for the corrected-protocol's 2,500 non-holdout,
hash-deduplicated images (`data/splits/cv_folds.csv`, repeat 0). The other
seven conditions each have their own, slightly larger, native row count.
The four demo images have to come from the *intersection* of all eight —
otherwise the sequence would have a gap at the one row that is supposed to
be its centrepiece.

In [ ]:
manifest = demo.step_1_manifest(cfg)
eligible, features, boxes, corrected_manifest_r0, coverage = demo.step_2_eligible_images(cfg, manifest)

print(f"full corpus: {len(manifest)} images")
coverage

In [ ]:
frozen_arms = demo.compute_all_frozen_arms(cfg, manifest, features)
cnn_table, _ = demo.cnn_oof(cfg, manifest)

print(f"{'arm':16s} {'n_rows':>8s}  fold-split random_state")
for arm_name, result in frozen_arms.items():
    print(f"{arm_name:16s} {len(result.path_order):8d}  {result.repeat_seed}")
print(f"{'cnn_corrected':16s} {len(cnn_table):8d}  (cv_folds.csv, repeat {cfg.repeat_index}, precomputed)")

### 2.2 Selection code and seed

Two autistic, two non_autistic, drawn at random from the eligible pool with
a fixed seed. The one departure from an unconstrained draw: if every one of
the four images a draw produces happens to be correctly classified by the
intact-arm probe, that draw is discarded and the next draw from the same
seeded stream is tried — a demo where every prediction is confidently
correct looks staged, and the task this notebook exists for is to show
prediction behaviour honestly, not flatteringly. This is the *only*
selection criterion beyond "at random, fixed seed"; the exact code:

In [ ]:
print(inspect.getsource(demo.select_demo_images))

In [ ]:
selected, n_attempts = demo.select_demo_images(cfg, eligible, frozen_arms["intact"])

print(f"selection_seed = {cfg.selection_seed}")
if n_attempts == 1:
    print("accepted on the very first, unconstrained draw from the seeded stream "
          "(it already contained a misclassified image).")
else:
    print(f"the first {n_attempts - 1} draw(s) were all-correct and discarded; "
          f"draw #{n_attempts} contained a misclassified image and was accepted.")
selected[["path", "filename", "split", "class_label"]]

In [ ]:
proba_by_path = dict(zip(frozen_arms["intact"].path_order, frozen_arms["intact"].proba))
y_by_path = dict(zip(frozen_arms["intact"].path_order, frozen_arms["intact"].y))

print(f"{'file':40s} {'true class':13s} {'intact P(autistic)':>19s}  {'predicted':>12s}  correct?")
for path, cls in zip(selected["path"], selected["class_label"]):
    p = proba_by_path[path]
    predicted = "autistic" if p >= cfg.decision_threshold else "non_autistic"
    correct = "yes" if predicted == cls else "NO -- misclassified"
    print(f"{Path(path).name:40s} {cls:13s} {p:19.3f}  {predicted:>12s}  {correct}")

### 2.3 Are these four images ordinary, or curated?

For every condition the demo sequence shows a probability for, here is that
probability's distribution across *every* image the condition covers, with
the four demo images marked. If the demo images were cherry-picked to look
good, they would cluster at the distribution's edges; if they are an
ordinary draw, they should scatter through the bulk of it.

In [ ]:
demo_paths = selected["path"].tolist()
demo_classes = dict(zip(selected["path"], selected["class_label"]))
demo_colors = {"autistic": "tab:red", "non_autistic": "tab:blue"}

conditions_for_typicality = [
    ("intact", "Intact photograph"),
    ("blur_sigma32", "Blurred sigma=32"),
    ("blur_sigma64", "Blurred sigma=64"),
    ("face_masked", "Face masked out"),
    ("face_crop", "Face crop only"),
    ("named_features", "Named features only"),
    ("facial_shape", "Facial shape only"),
]

fig, axes = plt.subplots(2, 4, figsize=(19, 8))
axes = axes.ravel()

for ax, (arm_name, label) in zip(axes, conditions_for_typicality):
    result = frozen_arms[arm_name]
    ax.hist(result.proba, bins=40, color="lightgray", edgecolor="white")
    for path in demo_paths:
        idx = int(np.where(result.path_order == path)[0][0])
        ax.axvline(result.proba[idx], color=demo_colors[demo_classes[path]], linewidth=2)
    ax.set_title(label, fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_xlabel("out-of-fold P(autistic)")

ax = axes[7]
ax.hist(cnn_table["proba"], bins=40, color="lightgray", edgecolor="white")
for path in demo_paths:
    val = cnn_table.loc[cnn_table["path"] == path, "proba"].iloc[0]
    ax.axvline(val, color=demo_colors[demo_classes[path]], linewidth=2)
ax.set_title("Fine-tuned CNN", fontsize=10)
ax.set_xlim(0, 1)
ax.set_xlabel("out-of-fold P(autistic)")

legend_elems = [
    Line2D([0], [0], color="tab:red", lw=2, label="demo image (autistic)"),
    Line2D([0], [0], color="tab:blue", lw=2, label="demo image (non_autistic)"),
]
fig.legend(handles=legend_elems, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(
    "Out-of-fold P(autistic) across every image each condition covers, four demo images marked",
    y=1.02,
)
fig.tight_layout()
plt.show()

In [ ]:
def _percentile_rank(value, array):
    return float((np.asarray(array) < value).mean() * 100)

rows = []
for arm_name, label in conditions_for_typicality:
    result = frozen_arms[arm_name]
    for path in demo_paths:
        idx = int(np.where(result.path_order == path)[0][0])
        val = float(result.proba[idx])
        rows.append(
            {
                "condition": label,
                "file": Path(path).name,
                "class_label": demo_classes[path],
                "probability": val,
                "percentile": _percentile_rank(val, result.proba),
            }
        )
for path in demo_paths:
    val = float(cnn_table.loc[cnn_table["path"] == path, "proba"].iloc[0])
    rows.append(
        {
            "condition": "Fine-tuned CNN",
            "file": Path(path).name,
            "class_label": demo_classes[path],
            "probability": val,
            "percentile": _percentile_rank(val, cnn_table["proba"].to_numpy()),
        }
    )
typicality = pd.DataFrame(rows)

extreme = typicality.loc[(typicality["percentile"] < 5) | (typicality["percentile"] > 95)]
if len(extreme):
    print("Demo images sitting in the extreme 5% of a condition's distribution "
          "(stated plainly here, not a reason to reselect):")
    print(extreme.to_string(index=False))
else:
    print("No demo image sits in the extreme 5% of any condition's distribution: "
          "all four are ordinary draws from their respective conditions.")
typicality.pivot(index=["file", "class_label"], columns="condition", values="percentile").round(1)

**One more thing these histograms show, unprompted.** The five
embedding-based conditions (intact, both blurs, masked, crop, CNN) are
sharply U-shaped: most images sit near 0 or near 1, hardly any land in the
middle. The two feature-based conditions (named features, facial shape —
a handful of tabular numbers, not a 2048-d embedding) are smoothly,
unimodally distributed instead. A high-dimensional embedding behind an
L2-logistic probe tends to push most points close to its decision boundary
in *logit* space but far from it in *probability* space, i.e. confidently
to one side or the other — this is a property of the classifier, not
evidence the model is more "sure" about those images. Keep this in mind for
Part 5: it is *why* a single image's score can jump sharply between two
conditions that otherwise look similar.

## 3. Out-of-fold assertions

Every probability this notebook is about to display must have come from a
model that never saw that image during training. This is not assumed: each
frozen-probe arm's claimed fold is independently re-derived from its
`StratifiedKFold` seed and checked against the image's row index; the CNN's
claimed fold is checked against `data/splits/cv_folds.csv`'s own repeat-0
fold assignment for that image. Every check below is a real `assert` in
`experiments/demo/pipeline.py` — a failure here would raise, not print.

In [ ]:
demo_table = demo.assemble_demo_table(selected, frozen_arms, cnn_table)

condition_short = {
    "Intact photograph": "intact",
    "Blurred sigma=32": "blur32",
    "Blurred sigma=64": "blur64",
    "Face masked out": "masked",
    "Face crop only": "crop",
    "Named features only": "named",
    "Facial shape only": "shape",
    "Fine-tuned CNN": "CNN",
}
header = f"{'file':40s} {'class':13s} " + " ".join(f"{c:>7s}" for c in condition_short.values())
print(header)

for path, cls in zip(selected["path"], selected["class_label"]):
    fold_by_condition = {}
    for arm_name, result in frozen_arms.items():
        fold = demo.assert_frozen_arm_out_of_fold(cfg, arm_name, result, path)
        fold_by_condition[demo.ARM_TO_CONDITION[arm_name]] = fold
    fold_by_condition["Fine-tuned CNN"] = demo.assert_cnn_out_of_fold(cnn_table, corrected_manifest_r0, path)

    row = " ".join(f"{fold_by_condition[c]:7d}" for c in condition_short)
    print(f"{Path(path).name:40s} {cls:13s} {row}")

print("\nAll assertions passed: every column above names the fold that held that "
      "image out for that condition; the model scoring that image never trained on it.")

## 4. The sequence: one row per image, one column per condition

The picture degrades to nothing recognisable, left to right. The number in
each panel's title is the model's out-of-fold probability that this
photograph shows an autistic child — labelled "model probability," never
"risk" or "likelihood," because it is a statement about the model's output,
not a clinical claim about the child.

In [ ]:
boxes_by_path = demo.box_lookup(boxes)
condition_labels = list(demo.CONDITION_ORDER)
n_rows = len(selected)

fig, axes = plt.subplots(n_rows, len(condition_labels), figsize=(3.0 * len(condition_labels), 3.2 * n_rows))

for row_i, (_, drow) in enumerate(selected.iterrows()):
    path = drow["path"]
    feat_row = features.loc[features["path"] == path].iloc[0]
    box = boxes_by_path[path]
    shape_points = demo.facial_shape_points(path, cfg)
    swatch = np.tile(demo.named_feature_swatch(feat_row), (50, 50, 1))

    panel_content = {
        "Intact photograph": demo.geometry_image(path, cfg),
        "Blurred sigma=32": demo.blurred_image(path, 32, cfg),
        "Blurred sigma=64": demo.blurred_image(path, 64, cfg),
        "Face masked out": demo.masked_image(path, box, cfg),
        "Face crop only": demo.cropped_image(path, box, cfg),
        "Named features only": swatch,
        "Facial shape only": None,  # drawn separately below, not an image
        "Fine-tuned CNN": demo.geometry_image(path, cfg),
    }

    for col_i, condition in enumerate(condition_labels):
        ax = axes[row_i, col_i]
        prob = demo_table.loc[
            (demo_table["path"] == path) & (demo_table["condition"] == condition), "probability"
        ].iloc[0]
        title_color = "crimson" if prob >= cfg.decision_threshold else "steelblue"

        if condition == "Facial shape only":
            ax.scatter(shape_points[:, 0], -shape_points[:, 1], s=1.5, color="lightgray")
            key_idx = [33, 133, 362, 263, 61, 291, 1, 234, 454]  # eyes, mouth corners, nose tip, cheeks
            ax.scatter(shape_points[key_idx, 0], -shape_points[key_idx, 1], s=14, color="black")
            ax.set_aspect("equal")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
        elif condition == "Named features only":
            ax.imshow(panel_content[condition])
            ax.text(
                25, 25,
                f"{int(feat_row['width'])}x{int(feat_row['height'])}px\n"
                f"{int(feat_row['file_size_bytes'])/1024:.0f} KB",
                ha="center", va="center", fontsize=8,
                color="white" if feat_row["mean_luminance"] < 128 else "black",
            )
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            ax.imshow(panel_content[condition])
            ax.set_xticks([])
            ax.set_yticks([])

        title = f"{condition}\np={prob:.2f}" if row_i == 0 else f"p={prob:.2f}"
        ax.set_title(title, fontsize=9, color=title_color)

    axes[row_i, 0].set_ylabel(f"{drow['class_label']}\n{Path(path).name}", fontsize=8)

fig.suptitle(
    "Model probability of \"autistic\" as the photograph degrades to nothing recognisable "
    "(red title >= 0.5, blue < 0.5)",
    y=1.0,
)
fig.tight_layout()
plt.show()

## 5. The slope chart: does a photograph's score hold steady?

Same eight conditions, now as one line per demo image, so the whole
trajectory is visible at once. Read it before deciding what it shows —
the honest answer is in the markdown cell right after it, not asserted
here first.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5))
x_positions = np.arange(len(condition_labels))

for _, drow in selected.iterrows():
    path = drow["path"]
    probs = [
        demo_table.loc[(demo_table["path"] == path) & (demo_table["condition"] == c), "probability"].iloc[0]
        for c in condition_labels
    ]
    ax.plot(
        x_positions, probs, marker="o", color=demo_colors[drow["class_label"]], alpha=0.85,
        label=f"{drow['class_label']} -- {Path(path).name}",
    )

ax.axhline(cfg.decision_threshold, color="gray", linestyle="--", linewidth=1, label=f"decision threshold ({cfg.decision_threshold})")
ax.set_xticks(x_positions)
ax.set_xticklabels(condition_labels, rotation=30, ha="right")
ax.set_ylabel('out-of-fold model probability of "autistic"')
ax.set_ylim(-0.02, 1.02)
ax.set_title("Probability across the sequence, one line per demo image")
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
plt.show()

**What this chart actually shows, for these four images: not a flat
line.** Each demo image's score swings substantially from one condition to
the next -- including between the intact photograph and a mild blur of it,
two conditions whose embeddings are almost the same picture. That is the
saturation effect flagged at the end of Part 2: five of the eight
conditions push most images close to 0 or 1, so a small shift in an image's
position relative to that condition's decision boundary is often enough to
flip which extreme it lands near. The two feature-based conditions (named
features, facial shape) move far more gently, because their probes are not
saturated the same way.

**What this does and does not mean.** It does not mean these classifiers
are unreliable in aggregate -- every arm's own experiment (and the
typicality histograms above) shows each one separating the two classes
above chance, consistently, across thousands of images. It does mean a
*single* image's predicted probability under any one condition is not a
stable property of that child carried forward from condition to condition
-- it is that condition's classifier's answer, on that condition's own
representation, and a different representation of the same photograph can
give a genuinely different answer. Whatever is driving classification here
is diffuse and representation-specific, not one consistent signal riding
along with the child through every transformation of their picture -- which
is itself an argument against reading any of these numbers as evidence
about the child, individually.

## 6. The photographic counterfactual

Take a non_autistic image and change only *photographic* properties —
never content, never framing: downscale then upscale back, re-encode at a
lower JPEG quality, soften slightly. The child in the photograph is
identical before and after.

**Which image, and why.** The displayed image is the non_autistic image
with the highest cached `sharpness` value — an objective, reproducible
rule ("in focus" is the core of "studio-style portrait"), not a visual
pick from the corpus.

**Scoring.** Both the original and the degraded embedding are scored by
that image's *own* out-of-fold intact-arm probe (the same fold model used
everywhere else in this notebook) — a model that never saw this image,
original or degraded, during training.

In [ ]:
studio_path = demo.select_studio_portrait(features)
print("displayed counterfactual image:", Path(studio_path).name)

batch = demo.select_counterfactual_batch(cfg, features, studio_path)
print(f"flip-rate batch: {len(batch)} non_autistic images (selection_seed={cfg.selection_seed}), "
      f"including the displayed one")

counterfactual_results = demo.counterfactual_flip_test(cfg, batch, frozen_arms["intact"])
flip_rate = counterfactual_results["crossed_threshold"].mean()
n_crossed = int(counterfactual_results["crossed_threshold"].sum())

print(f"\ndegradation: downsample to {cfg.counterfactual_downsample_size}px and back, "
      f"JPEG quality={cfg.counterfactual_jpeg_quality}, sharpness x{cfg.counterfactual_sharpness_factor}")
print(f"flip rate across the batch: {flip_rate:.1%} ({n_crossed}/{len(counterfactual_results)} images "
      f"crossed the {cfg.decision_threshold} decision threshold)")

studio_row = counterfactual_results.loc[counterfactual_results["path"] == studio_path].iloc[0]
print(f"\ndisplayed image: p_before={studio_row['p_before']:.3f} -> p_after={studio_row['p_after']:.3f} "
      f"({'CROSSED' if studio_row['crossed_threshold'] else 'did not cross'} the threshold)")

In [ ]:
original = demo.geometry_image(studio_path, cfg)
degraded = demo.degrade_photo(original, cfg)

fig, axes = plt.subplots(1, 2, figsize=(9, 5.6))
axes[0].imshow(original)
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_title(f"before\np={studio_row['p_before']:.3f}", fontsize=10)

axes[1].imshow(degraded)
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].set_title(
    f"after: downscale+upscale, JPEG q={cfg.counterfactual_jpeg_quality}, "
    f"sharpness x{cfg.counterfactual_sharpness_factor}\np={studio_row['p_after']:.3f}",
    fontsize=10,
)
fig.suptitle("Same child, same framing -- only the photograph's technical character moved", y=1.06)
fig.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for _, r in counterfactual_results.iterrows():
    color = "tab:red" if r["crossed_threshold"] else "lightgray"
    ax.plot([0, 1], [r["p_before"], r["p_after"]], color=color, alpha=0.7, linewidth=1, zorder=1)

ax.scatter(np.zeros(len(counterfactual_results)), counterfactual_results["p_before"], color="black", s=12, zorder=3)
ax.scatter(np.ones(len(counterfactual_results)), counterfactual_results["p_after"], color="black", s=12, zorder=3)
ax.axhline(cfg.decision_threshold, color="gray", linestyle="--", linewidth=1)
ax.set_xlim(-0.15, 1.15)
ax.set_xticks([0, 1])
ax.set_xticklabels(["before", "after"])
ax.set_ylabel('out-of-fold P("autistic"), intact-arm probe')
ax.set_title(
    f"{len(counterfactual_results)} non_autistic images: {flip_rate:.1%} cross the decision threshold\n"
    "after a purely photographic degradation (red = crossed)"
)
fig.tight_layout()
plt.show()

**Caveat, stated plainly.** A degraded image is out of the distribution
the intact-arm probe was trained on, so this result does not show that the
model would call this specific child autistic in a genuine, differently
-acquired snapshot — it shows that the model's prediction is highly
sensitive to *photographic* properties (resolution, compression, sharpness)
that have nothing to do with facial appearance. That is still the point:
whatever signal the intact-image arm's probe is picking up, a meaningful
share of it is not facial phenotype, because facial phenotype did not
change here.

## 7. What you just watched

Four photographs went through eight conditions each, from a full
photograph down to a blur, a solid colour patch, eleven geometric
measurements, and nothing at all — and every one of those thirty-two
probabilities came from a model that had never seen that image during
training, checked by an explicit assertion, not assumed. Rather than
holding steady, an individual image's score often swung sharply between
adjacent conditions, including between the intact photograph and a barely
-different blur of it — a real finding, not a flat line, and traced in
Part 5 to how saturated most of these classifiers' outputs are, not to any
instability in their aggregate accuracy. A purely photographic edit —
downscale, recompress, soften, with the child and the framing untouched —
still flipped a large share of a separate batch of images across the
decision threshold, which is a statement about what the model is sensitive
to, not about the children in the photographs. Put together, the aggregate
tables in the other six notebooks and this one photograph-at-a-time
sequence are telling a consistent story from two different angles: whatever
the intact-image arm is scoring is diffuse, representation-dependent, and
not the stable facial phenotype a clinical reading of these numbers would
require.